### Vollgeladen – Lösung Schritt für Schritt

*40. Bundeswettbewerb Informatik (2021), 1. Runde, Aufgabe 2*

Familie Meier reist mit dem Auto zu Verwandten in Portugal. Die Reiseroute steht fest, und ohne Übernachtungen wird es nicht gehen.
Die Meiers haben deshalb zahlreiche Hotels entlang der Route im Internet angesehen und für jedes eine Bewertung ermittelt.

Die Geschwister Lara und Paul dürfen auswählen, in welchen Hotels übernachtet wird. Im schon etwas betagten Auto der Meiers können die
beiden ihre Handys nicht aufladen. Deshalb können sie pro Tag maximal sechs Stunden lang fahren. Insgesamt darf die Reise höchstens
fünf Tage dauern.

Unter diesen Bedingungen wollen Lara und Paul möglichst schöne Hotels auswählen: Die niedrigste Bewertung eines Hotels ihrer Auswahl
soll so hoch wie möglich sein.

#### Aufgabe 2

Schreibe ein Programm, das die Auswahl der Hotels übernimmt. Dein Programm soll die Angaben zu den Hotels, nämlich ihre Positionen
entlang der Route und ihre Bewertungen, einlesen und eine Auswahl nach den Wünschen von Lara und Paul ausgeben.

Wende dein Programm mindestens auf alle Beispiele an, die du auf den BWINF-Webseiten findest, und dokumentiere die Ergebnisse.

#### Beispieldaten (`beispieldaten/hotels1.txt` bis `hotels5.txt`)

- 1. Zeile: Anzahl n der Hotels
- 2. Zeile: Gesamtfahrzeit in Minuten
- je Hotel eine Zeile: Entfernung vom Start (in Fahrminuten) und Bewertung

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Zielwert raten und prüfen:** Statt direkt „die beste Auswahl“ zu suchen, fragt man einfacher: „Geht die Reise, wenn ich nur Hotels mit
  Bewertung mindestens `x` benutze?“ Die Antwort ist ja oder nein, und das ist viel leichter zu berechnen.
- **Gierige Wahl:** Wer weit kommt, hat es am nächsten Tag nicht schwerer. Also fährt man jeden Tag so weit wie möglich.
- **Lineare Suche:** Die möglichen Bewertungen von oben nach unten durchprobieren, bis eine geht.
- **Binäre Suche:** Wenn „geht für `x`“ auch für jedes kleinere `x` gilt, kann man jeweils die Hälfte der Kandidaten verwerfen.

**Nützliche Funktionen und Bibliotheken**
- `set(...)` entfernt doppelte Werte, `sorted(..., reverse=True)` sortiert absteigend.
- Eine Liste aus Paaren (Tupeln) lässt sich mit `sort()` sortieren: Python sortiert nach dem ersten Eintrag, bei Gleichstand nach dem zweiten.

## Die Idee

Eine Auswahl von Hotels ist **gültig**, wenn

- höchstens 4 Hotels (4 Übernachtungen = 5 Tage) gewählt werden,
- das erste Hotel höchstens 360 Minuten (6 Stunden) vom Start entfernt ist,
- jedes weitere Hotel höchstens 360 Minuten nach dem vorigen kommt und
- das Ziel höchstens 360 Minuten vom letzten Hotel entfernt ist.

Gesucht ist die gültige Auswahl, deren **schlechtestes** Hotel möglichst gut bewertet ist.

**Schritt A: ein leichteres Problem.** Wir fragen: *„Gibt es überhaupt eine gültige Auswahl, wenn wir nur Hotels mit Bewertung
mindestens `x` benutzen dürfen?“* Dazu werfen wir alle Hotels mit einer Bewertung unter `x` weg. Aus den übrigen fahren wir **gierig**:
Jeden Tag fahren wir bis zum **am weitesten entfernten** Hotel, das noch in 360 Minuten erreichbar ist. Das kann nie schaden. Was man
heute weiter fährt, muss man morgen nicht mehr fahren, und morgen kann man von dort aus noch immer jedes Hotel erreichen, das man auch von
einem näheren Hotel aus erreicht hätte, und oft sogar mehr. Sobald das Ziel in Reichweite ist, sind wir fertig. Reichen 4 Übernachtungen nicht
oder gibt es kein Hotel mehr in Reichweite, geht es mit diesem `x` nicht.

**Schritt B: die beste Bewertung finden.** Als Mindestbewertung kommt nur eine Zahl in Frage, die bei einem Hotel tatsächlich vorkommt. Wir probieren sie
von der höchsten abwärts durch. Die erste, für die Schritt A „geht“ sagt, ist die Antwort. Das ist die **lineare Suche**. Weil mit einem kleineren `x` mehr Hotels
übrig bleiben, gilt: Geht es für `x`, so geht es auch für jedes kleinere. Deshalb funktioniert auch die **binäre Suche**, die viel weniger Prüfungen braucht.

**Zwei Feinheiten**, auf die die Bewertungskriterien hinweisen: Es können mehrere Hotels an derselben Position stehen (in `hotels2.txt` bei 341), mit verschiedenen
Bewertungen. Das ist kein Problem, weil wir jedes Hotel einzeln nach seiner Bewertung filtern. Und wir dürfen höchstens 4 Übernachtungen benutzen, auch wenn mehr
eine bessere Bewertung brächten (Beispiel `hotels3.txt`).

## Schritt 1: Einlesen

Wir lesen die Anzahl, die Gesamtfahrzeit und die Hotels ein. Jedes Hotel wird ein Paar `(Position, Bewertung)`. Die Paare sortieren wir nach der Position,
denn das gierige Verfahren geht die Hotels von vorn nach hinten durch.

In [ ]:
def lies_hotels(dateiname):
    with open(dateiname, encoding='utf-8-sig') as f:
        zeilen = [zeile.split() for zeile in f.read().splitlines() if zeile.strip()]
    gesamt = int(zeilen[1][0])
    hotels = [(int(pos), float(bew)) for pos, bew in zeilen[2:]]
    hotels.sort()
    return gesamt, hotels


gesamt, hotels = lies_hotels('beispieldaten/hotels1.txt')
print('Gesamtfahrzeit:', gesamt, 'Minuten, Hotels:', len(hotels))
print(hotels)

Gesamtfahrzeit: 1680 Minuten, Hotels: 12
[(12, 4.3), (326, 4.8), (347, 2.7), (359, 2.6), (553, 3.6), (590, 0.8), (687, 4.4), (1007, 2.8), (1008, 2.6), (1321, 2.1), (1360, 2.8), (1411, 3.3)]


## Schritt 2: Geht die Reise mit Mindestbewertung `x`?

`reise_moeglich(hotels, gesamt, x)` liefert die gewählten Hotels als Liste, oder `None`, wenn es nicht geht.

- `gut` enthält nur die Hotels mit Bewertung mindestens `x`, weiterhin nach der Position sortiert.
- `pos` ist die Position, an der wir gerade stehen (am Anfang 0).
- Wir wiederholen höchstens 4-mal: Ist das Ziel in höchstens 360 Minuten erreichbar (`gesamt - pos <= 360`), sind wir fertig. Sonst suchen wir unter allen
  guten Hotels mit `pos < position <= pos + 360` das letzte, also das am weitesten entfernte. Gibt es keins, geht es nicht.
- Nach den 4 Übernachtungen muss das Ziel in Reichweite sein.

Die Anzahl der Übernachtungen ist ein Parameter `naechte` (4 laut Aufgabe), damit wir später ausprobieren können, was bei mehr Übernachtungen möglich wäre.

In [ ]:
MAX_MINUTEN = 360          # 6 Stunden Fahrt pro Tag


def reise_moeglich(hotels, gesamt, x, naechte=4):
    gut = [h for h in hotels if h[1] >= x]
    pos = 0
    gewaehlt = []
    for nacht in range(naechte):
        if gesamt - pos <= MAX_MINUTEN:
            return gewaehlt                      # Ziel ist erreichbar
        weitestes = None
        for hotel in gut:
            if pos < hotel[0] <= pos + MAX_MINUTEN:
                weitestes = hotel                # gut ist sortiert: das letzte passende ist das weiteste
        if weitestes is None:
            return None                          # kein Hotel in Reichweite
        gewaehlt.append(weitestes)
        pos = weitestes[0]
    if gesamt - pos <= MAX_MINUTEN:
        return gewaehlt
    return None


print(reise_moeglich(hotels, gesamt, 0.0))
print(reise_moeglich(hotels, gesamt, 2.7))
print(reise_moeglich(hotels, gesamt, 2.8))

[(359, 2.6), (687, 4.4), (1008, 2.6), (1360, 2.8)]
[(347, 2.7), (687, 4.4), (1007, 2.8), (1360, 2.8)]
None


Ohne Einschränkung (`x = 0.0`) wählt das Verfahren immer das weiteste Hotel, die Bewertungen sind dort beliebig (2.6 ist die kleinste). Mit `x = 2.7` geht es noch,
mit `x = 2.8` nicht mehr: Die Antwort `None` heißt, dass es mit den Hotels ab 2.8 keine gültige Auswahl gibt.

## Schritt 3: Die beste Bewertung (lineare Suche)

Die möglichen Mindestbewertungen sind die Bewertungen, die in der Datei vorkommen. `set` entfernt doppelte, `sorted(..., reverse=True)` ordnet sie von der besten
zur schlechtesten. Wir prüfen sie der Reihe nach und halten bei der ersten an, für die die Reise geht. Dabei zählen wir die Prüfungen.

Eine Reise unter 6 Stunden braucht gar keine Übernachtung. Diesen Sonderfall fangen wir vorab ab.

In [ ]:
def beste_linear(hotels, gesamt):
    """Liefert (Mindestbewertung, gewählte Hotels, Anzahl Prüfungen)."""
    if gesamt <= MAX_MINUTEN:
        return None, [], 0                       # keine Übernachtung nötig
    bewertungen = sorted(set(b for pos, b in hotels), reverse=True)
    pruefungen = 0
    for x in bewertungen:
        pruefungen = pruefungen + 1
        gewaehlt = reise_moeglich(hotels, gesamt, x)
        if gewaehlt is not None:
            return x, gewaehlt, pruefungen
    return -1, None, pruefungen                  # auch mit allen Hotels geht es nicht


x, gewaehlt, pruefungen = beste_linear(hotels, gesamt)
print('beste Mindestbewertung:', x, ' Prüfungen:', pruefungen)
print(gewaehlt)

beste Mindestbewertung: 2.7  Prüfungen: 7
[(347, 2.7), (687, 4.4), (1007, 2.8), (1360, 2.8)]


## Schritt 4: Schneller mit binärer Suche

Die Liste `bewertungen` ist sortiert, diesmal aufsteigend. Ganz links stehen die Werte, bei denen es sicher geht, ganz rechts die, bei denen es nicht geht.
Wir suchen die **Grenze** dazwischen. Dazu halten wir zwei Grenzen `links` und `rechts` fest:

- `bewertungen[links]` geht sicher (am Anfang die kleinste Bewertung, denn dann sind alle Hotels erlaubt).
- Alles ab `rechts` geht nicht (am Anfang `len(bewertungen)`).

In jeder Runde prüfen wir die Mitte. Geht sie, wird sie die neue linke Grenze, sonst die neue rechte. Wenn `links + 1 == rechts` gilt, ist `bewertungen[links]` die beste.
Ist schon die kleinste Bewertung nicht gültig, kann man mit keinem Hotel fahren und es gibt keine Lösung.

In [ ]:
def beste_binaer(hotels, gesamt):
    """Liefert (Mindestbewertung, gewählte Hotels, Anzahl Prüfungen)."""
    if gesamt <= MAX_MINUTEN:
        return None, [], 0
    bewertungen = sorted(set(b for pos, b in hotels))
    pruefungen = 1
    links = 0
    rechts = len(bewertungen)
    if reise_moeglich(hotels, gesamt, bewertungen[0]) is None:
        return -1, None, pruefungen
    while links + 1 < rechts:
        mitte = (links + rechts) // 2
        pruefungen = pruefungen + 1
        if reise_moeglich(hotels, gesamt, bewertungen[mitte]) is not None:
            links = mitte
        else:
            rechts = mitte
    x = bewertungen[links]
    return x, reise_moeglich(hotels, gesamt, x), pruefungen


x, gewaehlt, pruefungen = beste_binaer(hotels, gesamt)
print('beste Mindestbewertung:', x, ' Prüfungen:', pruefungen)
print(gewaehlt)

beste Mindestbewertung: 2.7  Prüfungen: 5
[(347, 2.7), (687, 4.4), (1007, 2.8), (1360, 2.8)]


## Schritt 5: Alles zusammen

`loese` liest eine Datei ein, löst sie mit der binären Suche und gibt die Ergebnisse aus: die Mindestbewertung und die Hotels mit Position und Bewertung.
Zur Kontrolle prüft die Funktion `pruefe` die Lösung unabhängig von der Suche (alle Abstände höchstens 6 Stunden, höchstens 4 Hotels, die kleinste Bewertung stimmt).
Außerdem vergleichen wir mit der linearen Suche und zählen die Prüfungen beider Verfahren.

In [ ]:
def pruefe(gewaehlt, gesamt, x):
    positionen = [0] + [pos for pos, b in gewaehlt] + [gesamt]
    ok = len(gewaehlt) <= 4
    for i in range(len(positionen) - 1):
        ok = ok and 0 <= positionen[i + 1] - positionen[i] <= MAX_MINUTEN
    ok = ok and min(b for pos, b in gewaehlt) == x
    return ok


def loese(dateiname):
    gesamt, hotels = lies_hotels(dateiname)
    x, gewaehlt, pruefungen = beste_binaer(hotels, gesamt)
    x_linear, gewaehlt_linear, pruefungen_linear = beste_linear(hotels, gesamt)
    print(dateiname.split('/')[-1], ':', len(hotels), 'Hotels, Gesamtfahrzeit', gesamt, 'Minuten')
    if x is None:
        print('  Keine Übernachtung nötig.')
    elif gewaehlt is None:
        print('  Die Reise ist mit höchstens 4 Übernachtungen nicht möglich.')
    else:
        assert x == x_linear and pruefe(gewaehlt, gesamt, x)
        print('  niedrigste Bewertung:', x)
        for i in range(len(gewaehlt)):
            print('  Hotel', i + 1, ': Position', gewaehlt[i][0], ' Bewertung', gewaehlt[i][1])
    print('  Prüfungen: binär', pruefungen, ', linear', pruefungen_linear)


for nr in range(1, 6):
    loese('beispieldaten/hotels' + str(nr) + '.txt')
    print()

hotels1.txt : 12 Hotels, Gesamtfahrzeit 1680 Minuten
  niedrigste Bewertung: 2.7
  Hotel 1 : Position 347  Bewertung 2.7
  Hotel 2 : Position 687  Bewertung 4.4
  Hotel 3 : Position 1007  Bewertung 2.8
  Hotel 4 : Position 1360  Bewertung 2.8
  Prüfungen: binär 5 , linear 7

hotels2.txt : 25 Hotels, Gesamtfahrzeit 1737 Minuten
  niedrigste Bewertung: 2.3
  Hotel 1 : Position 341  Bewertung 2.3
  Hotel 2 : Position 700  Bewertung 3.0
  Hotel 3 : Position 1053  Bewertung 4.8
  Hotel 4 : Position 1380  Bewertung 5.0
  Prüfungen: binär 6 , linear 10

hotels3.txt : 500 Hotels, Gesamtfahrzeit 1793 Minuten
  niedrigste Bewertung: 0.3
  Hotel 1 : Position 360  Bewertung 1.3
  Hotel 2 : Position 717  Bewertung 0.3
  Hotel 3 : Position 1076  Bewertung 3.8
  Hotel 4 : Position 1433  Bewertung 1.7
  Prüfungen: binär 7 , linear 48

hotels4.txt : 500 Hotels, Gesamtfahrzeit 1510 Minuten
  niedrigste Bewertung: 4.6
  Hotel 1 : Position 340  Bewertung 4.6
  Hotel 2 : Position 676  Bewertung 4.6
  Hotel

## Auswertung

Die Ergebnisse stimmen mit der Tabelle der offiziellen Lösungshinweise überein: niedrigste Bewertung 2.7, 2.3, 0.3, 4.6 und 5.0, und sogar dieselben Hotels.
(Eine andere Auswahl mit derselben niedrigsten Bewertung wäre ebenso richtig.)

- `hotels1.txt`: Das Hotel bei 326 (4.8) ist zwar am besten bewertet, aber das Verfahren fährt bis 347 (2.7), weil das weiter ist. Für die niedrigste Bewertung 2.7 reicht das.
- `hotels2.txt`: Bei Position 341 stehen Hotels mit 2.2 und 2.3. Weil nach der Bewertung gefiltert wird, kommt nur das mit 2.3 in Frage.
- `hotels3.txt`: Die niedrigste Bewertung ist nur 0.3. Mit mehr Übernachtungen ginge es besser (siehe unten), die Aufgabe erlaubt aber nur 4.
- `hotels5.txt`: Es gibt Hotels mit der Bewertung 5.0 an passenden Stellen, daher ist das Ergebnis 5.0, das Maximum.

Bei den Prüfungen gewinnt die binäre Suche meistens klar (`hotels3.txt`: 7 statt 48). Sie ist aber nicht immer besser: Wenn die Lösung bei den höchsten Bewertungen liegt, findet die lineare Suche sie gleich
(`hotels5.txt`: 1 Prüfung statt 7). Im schlechtesten Fall braucht die lineare Suche so viele Prüfungen, wie es verschiedene Bewertungen gibt, die binäre nur etwa deren Zweierlogarithmus.

## Probe: Was brächte eine fünfte Übernachtung?

Weil `naechte` ein Parameter ist, können wir es ausprobieren. Für `hotels3.txt` suchen wir mit 4, 5 und 6 Übernachtungen die höchste Mindestbewertung
(einfach alle Bewertungen absteigend durchprobieren):

In [ ]:
def beste_mit_naechten(hotels, gesamt, naechte):
    for x in sorted(set(b for pos, b in hotels), reverse=True):
        if reise_moeglich(hotels, gesamt, x, naechte) is not None:
            return x
    return None


gesamt3, hotels3 = lies_hotels('beispieldaten/hotels3.txt')
for naechte in [4, 5, 6]:
    print(naechte, 'Übernachtungen: höchste Mindestbewertung', beste_mit_naechten(hotels3, gesamt3, naechte))

4 Übernachtungen: höchste Mindestbewertung 0.3
5 Übernachtungen: höchste Mindestbewertung 4.8
6 Übernachtungen: höchste Mindestbewertung 4.8


Mit 5 Übernachtungen wäre die Mindestbewertung 4.8 statt 0.3. Wer das nicht beachtet und mehr Hotels zulässt, bekommt also ein falsches Ergebnis. Dafür gab es Punktabzug.

## Zum Schluss: eine Lösung mit MiniZinc

Hier passt MiniZinc gut: Die Aufgabe besteht aus **vier Auswahl-Entscheidungen** und ein paar Bedingungen, und das Ziel ist ein **Maximum eines Minimums**.
Wir beschreiben nur, was gelten soll. Die gierige Idee und die Suche über die Bewertungen übernimmt der Solver.

Die MiniZinc-Blöcke unten sind **kein Python**: Kopiere sie in die MiniZinc IDE (oder in den
[MiniZinc Playground](https://play.minizinc.org/)) und führe sie dort aus.

#### Die Daten

Das Modell bekommt die Positionen und Bewertungen als zwei Arrays. MiniZinc rechnet mit ganzen Zahlen, deshalb schreiben wir die Bewertungen
in Zehnteln (4.3 wird 43). Die Hotels sind nach der Position sortiert, das nutzt das Modell aus. Die Python-Zelle macht aus einer Beispieldatei den Inhalt der
Datendatei. Kopiere die Ausgabe in eine Datei `hotels1.dzn`.

In [ ]:
def als_dzn(dateiname):
    gesamt, hotels = lies_hotels(dateiname)
    positionen = [pos for pos, bew in hotels]
    zehntel = [round(bew * 10) for pos, bew in hotels]
    text = 'n = ' + str(len(hotels)) + ';\n'
    text = text + 'gesamt = ' + str(gesamt) + ';\n'
    text = text + 'position = ' + str(positionen) + ';\n'
    text = text + 'bewertung = ' + str(zehntel) + ';'
    return text


print(als_dzn('beispieldaten/hotels1.txt'))

n = 12;
gesamt = 1680;
position = [12, 326, 347, 359, 553, 590, 687, 1007, 1008, 1321, 1360, 1411];
bewertung = [43, 48, 27, 26, 36, 8, 44, 28, 26, 21, 28, 33];


#### Das Modell

Die vier Variablen `h[1]` bis `h[4]` sind die **Nummern** der gewählten Hotels. Sie dienen als Index in die Arrays (`position[h[k]]` ist die Position des k-ten Hotels).
Die Bedingungen übersetzen die vier Punkte der Gültigkeit. Mit `h[k] >= h[k-1]` bleiben die Hotels in der Reihenfolge der Route. Das Ziel ist
`maximize`, wobei `min(k in 1..4)(...)` die schlechteste Bewertung der vier ist.

```minizinc
% ----- Daten (aus der .dzn-Datei) -----
int: n;                                 % Anzahl der Hotels
int: gesamt;                            % Gesamtfahrzeit in Minuten
array[1..n] of int: position;           % Minuten vom Start, aufsteigend sortiert
array[1..n] of int: bewertung;          % Bewertung in Zehnteln

int: maximal = 360;                     % 6 Stunden pro Tag

% ----- Entscheidungsvariablen -----
% h[k] = Nummer des Hotels der k-ten Übernachtung.
% (Reichen weniger als 4 Nächte, wählt der Solver ein Hotel mehrfach; das schadet nicht.)
array[1..4] of var 1..n: h;

% ----- Bedingungen -----
constraint position[h[1]] <= maximal;                                  % erstes Hotel vom Start aus erreichbar
constraint forall(k in 2..4)(h[k] >= h[k-1]                            % Hotels in Reihenfolge der Route
                             /\ position[h[k]] - position[h[k-1]] <= maximal);
constraint gesamt - position[h[4]] <= maximal;                         % Ziel vom letzten Hotel aus erreichbar

% ----- Ziel: die schlechteste Bewertung so hoch wie möglich -----
var int: schlechteste = min(k in 1..4)(bewertung[h[k]]);
solve maximize schlechteste;

output ["schlechteste Bewertung: \(schlechteste)\n"] ++
       ["Hotel \(k): Position \(position[h[k]]), Bewertung \(bewertung[h[k]])\n" | k in 1..4];
```

Mit den Daten von `hotels1.txt`:

**Ausgabe:**

```
schlechteste Bewertung: 27
Hotel 1: Position 347, Bewertung 27
Hotel 2: Position 687, Bewertung 44
Hotel 3: Position 1007, Bewertung 28
Hotel 4: Position 1360, Bewertung 28
----------
==========
```

Das ist dieselbe Mindestbewertung (2.7) wie oben. Den Solver stören mehrere Hotels an einer Position nicht. Für `hotels5.txt` (1500 Hotels):

**Ausgabe:**

```
schlechteste Bewertung: 50
Hotel 1: Position 280, Bewertung 50
Hotel 2: Position 636, Bewertung 50
Hotel 3: Position 987, Bewertung 50
Hotel 4: Position 1271, Bewertung 50
----------
==========
```

Alle fünf Beispiele rechnet der Solver (Gecode) jeweils in deutlich unter einer Sekunde, auch das mit 1500 Hotels. Die Hotels können sich von denen der Python-Lösung unterscheiden
(MiniZinc liefert irgendeine optimale Auswahl, nicht unbedingt die mit den weitesten Hotels), die Mindestbewertung ist aber dieselbe.

*Warum nur vier Variablen?* Im Gegensatz zu vielen anderen Aufgaben ist hier die Zahl der Wahlentscheidungen klein und fest. Der Solver kann gut abschneiden: Er wählt
`h[1]` und prüft die Bedingungen, wählt `h[2]` und so weiter. Den Aufwand `n^4` des naiven Probierens vermeidet er, weil er Zweige früh verwirft, in denen eine Bedingung schon verletzt ist.